# Day 1 - Market Case and Research-Contract Layering

## tl;dr

**Day 1 gate: PASS.**

- Three explicit layers: HSBC `market_case`, `RC-L`, and `RC-A`.
- Deterministic cash-flow paths: **10/10 passed**.
- Maximum component identity error: **0.000e+00**.
- Known fair coupon **0.075000** recovered as
  **0.075000**.
- Failed criteria: **none**.

This is a contract-definition and deterministic-validation layer. It does not
claim that RC-A is the issuer HSBC security or a traded market price.

## Context & Methods

This notebook validates the contract architecture required by the current
project plan. It preserves the issuer HSBC note as the `market_case` and
adds two separate research contracts:

- **RC-L:** the Lee et al. reproduction contract;
- **RC-A:** a stylised, market-informed, continuously monitored KI contract.

### Key Assumptions

- The real HSBC security retains maturity-only knock-in monitoring.
- RC-L and RC-A are research contracts and cannot be described as the HSBC note.
- Continuous KI and discrete autocall are separate events.
- Under spot bumps, contractual initial references and absolute barriers stay fixed.
- RC-A coupon memory pays all accrued missed quarterly coupons when its coupon
  trigger is next satisfied.
- Issuer credit, funding, liquidity and dealer margin are outside GBM model value.
- Additive value identity:

  \[
  V=V_{coupon}+V_{early\ principal}+V_{surviving\ notional}+V_{KI\ loss}.
  \]

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import platform
import sys

import numpy as np
import pandas as pd
from scipy.optimize import brentq

pd.set_option("display.precision", 10)


def find_project_root():
    candidates = []
    override = os.environ.get("AP_PROJECT_ROOT")
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "config" / "core_project_config.json").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate core_project_config.json")


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


PROJECT_DIR = find_project_root()
CONFIG_FILE = PROJECT_DIR / "config" / "core_project_config.json"
CONFIG = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
SOURCE_FILE = PROJECT_DIR / CONFIG["market_data"]["relative_path"]
OUTPUT_DIR = PROJECT_DIR / "outputs" / "day1_contract_layers"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_HASH = sha256_file(SOURCE_FILE)
EXPECTED_HASH = CONFIG["market_data"]["sha256"].upper()
print(f"Project root: {PROJECT_DIR}")
print(f"Config schema: {CONFIG['schema_version']}")
print(f"Workbook hash match: {SOURCE_HASH == EXPECTED_HASH}")

## Data

### 1. Validate the three contract layers and economic conventions

In [ ]:
required_top_level = [
    "market_case",
    "research_contracts",
    "event_definitions",
    "bump_policy",
    "valuation_scope_exclusions",
    "component_schema",
]
missing_top_level = [key for key in required_top_level if key not in CONFIG]
assert not missing_top_level, f"Missing config sections: {missing_top_level}"
assert set(CONFIG["research_contracts"]) == {"RC-L", "RC-A"}
assert CONFIG["market_case"]["layer"] == "market_case"
assert all(
    contract["layer"] == "research_contract"
    for contract in CONFIG["research_contracts"].values()
)
assert CONFIG["market_case"]["knock_in_monitoring"] == "maturity only"
assert CONFIG["research_contracts"]["RC-L"]["knock_in"]["monitoring"] == "continuous"
assert CONFIG["research_contracts"]["RC-A"]["knock_in"]["monitoring"] == "continuous"
assert SOURCE_HASH == EXPECTED_HASH
assert all(item["initial_reference"] > 0 for item in CONFIG["underlyings"])

layer_summary = pd.DataFrame(
    [
        {
            "Layer": "market_case",
            "ID": CONFIG["market_case"]["id"],
            "Label": CONFIG["market_case"]["label"],
            "KI monitoring": CONFIG["market_case"]["knock_in_monitoring"],
            "Claim boundary": CONFIG["market_case"]["model_use"],
        },
        *[
            {
                "Layer": "research_contract",
                "ID": contract_id,
                "Label": contract["label"],
                "KI monitoring": contract["knock_in"]["monitoring"],
                "Claim boundary": contract["claim_boundary"],
            }
            for contract_id, contract in CONFIG["research_contracts"].items()
        ],
    ]
)
display(layer_summary)
display(pd.DataFrame(CONFIG["component_schema"]["field_definitions"].items(), columns=["Field", "Definition"]))

## Results

### 2. Run ten deterministic cash-flow paths

The tests cover early calls, missed/recovered memory coupons, no-KI maturity,
continuous-KI recovery, terminal KI loss, boundary equality and asymmetric
worst-of paths. Cash-flow components are discounted path by path.

In [ ]:
def present_value(amount, rate, time_years):
    return float(amount * math.exp(-rate * time_years))


def evaluate_rc_l_path(path, annual_coupon=0.06, rate=0.03):
    contract = CONFIG["research_contracts"]["RC-L"]
    principal = contract["principal"]
    times = np.asarray(contract["observation_times_years"], dtype=float)
    barriers = np.asarray(contract["autocall_barrier_ratios"], dtype=float)
    observations = np.asarray(path["observations"], dtype=float)
    interval_minima = np.asarray(path["interval_minima"], dtype=float)
    if observations.shape != (len(times), 3) or interval_minima.shape != observations.shape:
        raise ValueError("RC-L path arrays must be observations x 3")

    call_index = None
    for index, trigger in enumerate(barriers):
        if np.min(observations[index]) >= trigger:
            call_index = index
            break

    coupon_value = 0.0
    early_principal = 0.0
    surviving_notional = 0.0
    terminal_ki_loss = 0.0
    no_ki_redemption = 0.0
    ki_event = bool(np.min(interval_minima) <= contract["knock_in"]["barrier_ratio"])

    if call_index is not None:
        redemption_time = times[call_index]
        coupon_value = present_value(principal * annual_coupon * redemption_time, rate, redemption_time)
        early_principal = present_value(principal, rate, redemption_time)
        outcome = f"autocall_{call_index + 1}"
    else:
        maturity = times[-1]
        terminal_worst = float(np.min(observations[-1]))
        surviving_notional = present_value(principal, rate, maturity)
        if ki_event and terminal_worst < 1.0:
            terminal_ki_loss = present_value(principal * (terminal_worst - 1.0), rate, maturity)
            outcome = "maturity_ki_loss"
        else:
            coupon_value = present_value(principal * annual_coupon * maturity, rate, maturity)
            if not ki_event:
                no_ki_redemption = surviving_notional
                outcome = "maturity_no_ki"
            else:
                outcome = "maturity_ki_recovered"

    total = coupon_value + early_principal + surviving_notional + terminal_ki_loss
    return {
        "coupon_value": coupon_value,
        "early_redemption_principal": early_principal,
        "surviving_notional": surviving_notional,
        "no_ki_maturity_redemption": no_ki_redemption,
        "terminal_ki_loss": terminal_ki_loss,
        "total_value": total,
        "outcome": outcome,
        "continuous_ki": ki_event,
        "autocall_index": None if call_index is None else call_index + 1,
    }


def evaluate_rc_a_path(path, annual_coupon=0.0875, rate=0.03):
    contract = CONFIG["research_contracts"]["RC-A"]
    schedule = contract["observation_schedule"]
    times = np.asarray([item["time_years_act365"] for item in schedule], dtype=float)
    observations = np.asarray(path["observations"], dtype=float)
    interval_minima = np.asarray(path["interval_minima"], dtype=float)
    if observations.shape != (len(schedule), 3) or interval_minima.shape != observations.shape:
        raise ValueError("RC-A path arrays must be observations x 3")

    principal = contract["principal"]
    coupon_trigger = contract["coupon"]["trigger_ratio"]
    period_coupon = principal * annual_coupon / 4.0
    accrued_periods = 0
    coupon_value = 0.0
    early_principal = 0.0
    surviving_notional = 0.0
    terminal_ki_loss = 0.0
    no_ki_redemption = 0.0
    call_index = None

    for index, item in enumerate(schedule):
        worst = float(np.min(observations[index]))
        accrued_periods += 1
        if worst >= coupon_trigger:
            coupon_value += present_value(period_coupon * accrued_periods, rate, times[index])
            accrued_periods = 0
        if item["autocall"] and worst >= item["autocall_trigger_ratio"]:
            early_principal = present_value(principal, rate, times[index])
            call_index = index
            break

    ki_event = bool(np.min(interval_minima) <= contract["knock_in"]["barrier_ratio"])
    if call_index is not None:
        outcome = f"autocall_{call_index + 1}"
    else:
        maturity = times[-1]
        terminal_worst = float(np.min(observations[-1]))
        surviving_notional = present_value(principal, rate, maturity)
        if ki_event and terminal_worst < 1.0:
            terminal_ki_loss = present_value(principal * (terminal_worst - 1.0), rate, maturity)
            outcome = "maturity_ki_loss"
        elif not ki_event:
            no_ki_redemption = surviving_notional
            outcome = "maturity_no_ki"
        else:
            outcome = "maturity_ki_recovered"

    total = coupon_value + early_principal + surviving_notional + terminal_ki_loss
    return {
        "coupon_value": coupon_value,
        "early_redemption_principal": early_principal,
        "surviving_notional": surviving_notional,
        "no_ki_maturity_redemption": no_ki_redemption,
        "terminal_ki_loss": terminal_ki_loss,
        "total_value": total,
        "outcome": outcome,
        "continuous_ki": ki_event,
        "autocall_index": None if call_index is None else call_index + 1,
    }


def repeated_rows(values, rows):
    return np.tile(np.asarray(values, dtype=float), (rows, 1))


rc_l_rows = 6
rc_a_rows = len(CONFIG["research_contracts"]["RC-A"]["observation_schedule"])
deterministic_paths = [
    {"id": "L1", "contract": "RC-L", "observations": repeated_rows([0.95, 0.94, 0.93], rc_l_rows), "interval_minima": repeated_rows([0.80, 0.82, 0.81], rc_l_rows), "expected": "autocall_1"},
    {"id": "L2", "contract": "RC-L", "observations": np.array([[0.80,0.82,0.81],[0.92,0.91,0.93],[0.80,0.80,0.80],[0.80,0.80,0.80],[0.80,0.80,0.80],[0.80,0.80,0.80]]), "interval_minima": repeated_rows([0.70,0.72,0.71], rc_l_rows), "expected": "autocall_2"},
    {"id": "L3", "contract": "RC-L", "observations": repeated_rows([0.75,0.76,0.77], rc_l_rows), "interval_minima": repeated_rows([0.60,0.62,0.61], rc_l_rows), "expected": "maturity_no_ki"},
    {"id": "L4", "contract": "RC-L", "observations": repeated_rows([0.72,0.74,0.70], rc_l_rows), "interval_minima": np.vstack([repeated_rows([0.60,0.61,0.62], rc_l_rows-1), [0.44,0.58,0.60]]), "expected": "maturity_ki_loss"},
    {"id": "L5", "contract": "RC-L", "observations": np.vstack([repeated_rows([0.70,0.72,0.71], rc_l_rows-1), [1.05,1.02,1.01]]), "interval_minima": np.vstack([[0.45,0.70,0.72], repeated_rows([0.60,0.61,0.62], rc_l_rows-1)]), "expected": "autocall_6"},
    {"id": "A1", "contract": "RC-A", "observations": np.vstack([[0.80,0.82,0.81],[1.02,1.01,1.03],repeated_rows([0.90,0.91,0.92], rc_a_rows-2)]), "interval_minima": repeated_rows([0.80,0.81,0.82], rc_a_rows), "expected": "autocall_2"},
    {"id": "A2", "contract": "RC-A", "observations": np.vstack([[0.70,0.72,0.71],[1.04,1.02,1.03],repeated_rows([0.90,0.91,0.92], rc_a_rows-2)]), "interval_minima": repeated_rows([0.78,0.79,0.80], rc_a_rows), "expected": "autocall_2"},
    {"id": "A3", "contract": "RC-A", "observations": repeated_rows([0.90,0.88,0.86], rc_a_rows), "interval_minima": repeated_rows([0.80,0.82,0.81], rc_a_rows), "expected": "maturity_no_ki"},
    {"id": "A4", "contract": "RC-A", "observations": repeated_rows([0.60,0.68,0.65], rc_a_rows), "interval_minima": np.vstack([repeated_rows([0.80,0.82,0.81], rc_a_rows-1), [0.74,0.80,0.82]]), "expected": "maturity_ki_loss"},
    {"id": "A5", "contract": "RC-A", "observations": np.vstack([repeated_rows([0.85,0.83,0.82], rc_a_rows-1), [1.05,1.02,1.01]]), "interval_minima": np.vstack([[0.75,0.80,0.82], repeated_rows([0.82,0.83,0.84], rc_a_rows-1)]), "expected": "autocall_8"},
]

path_rows = []
for path in deterministic_paths:
    evaluator = evaluate_rc_l_path if path["contract"] == "RC-L" else evaluate_rc_a_path
    result = evaluator(path)
    additive_sum = sum(result[field] for field in CONFIG["component_schema"]["additive_components"])
    path_rows.append({
        "path_id": path["id"],
        "contract": path["contract"],
        "expected_outcome": path["expected"],
        **result,
        "additive_sum": additive_sum,
        "identity_error": result["total_value"] - additive_sum,
        "outcome_pass": result["outcome"] == path["expected"],
    })

deterministic_results = pd.DataFrame(path_rows)
display(deterministic_results)
assert len(deterministic_results) == 10
assert deterministic_results["outcome_pass"].all()
assert deterministic_results["identity_error"].abs().max() <= 1e-10

### 3. Recover a known fair coupon with a bracketed root solver

A fixed three-path RC-A portfolio is priced at a deliberately chosen annual
coupon of 7.5%. The solver sees only the target value and must recover the
original coupon from a wide bracket.

In [ ]:
root_paths = [path for path in deterministic_paths if path["contract"] == "RC-A"][:3]
root_weights = np.array([0.25, 0.35, 0.40])
known_coupon = 0.075


def deterministic_portfolio_value(annual_coupon):
    values = np.array([
        evaluate_rc_a_path(path, annual_coupon=annual_coupon)["total_value"]
        for path in root_paths
    ])
    return float(root_weights @ values)


artificial_target = deterministic_portfolio_value(known_coupon)
recovered_coupon = brentq(
    lambda coupon: deterministic_portfolio_value(coupon) - artificial_target,
    0.0,
    0.30,
    xtol=1e-14,
    rtol=1e-14,
)
root_error = recovered_coupon - known_coupon
fair_coupon_root_test = pd.DataFrame([
    {
        "known_coupon": known_coupon,
        "target_value": artificial_target,
        "recovered_coupon": recovered_coupon,
        "root_error": root_error,
        "absolute_residual": abs(deterministic_portfolio_value(recovered_coupon) - artificial_target),
        "solver": "scipy.optimize.brentq",
        "bracket": "[0.00, 0.30]",
        "pass": abs(root_error) <= 1e-10,
    }
])
display(fair_coupon_root_test)
assert bool(fair_coupon_root_test["pass"].iloc[0])

### 4. Day 1 gate and auditable outputs

In [ ]:
gate_summary = pd.DataFrame([
    {"criterion": "HSBC workbook hash and official references", "observed": f"hash_match={SOURCE_HASH == EXPECTED_HASH}; positive_refs={sum(item['initial_reference'] > 0 for item in CONFIG['underlyings'])}/3", "pass": SOURCE_HASH == EXPECTED_HASH and all(item["initial_reference"] > 0 for item in CONFIG["underlyings"])},
    {"criterion": "market_case, RC-L and RC-A are separate layers", "observed": ", ".join(layer_summary["ID"]), "pass": len(layer_summary) == 3 and layer_summary["Layer"].tolist().count("market_case") == 1},
    {"criterion": "continuous KI and discrete autocall separated", "observed": json.dumps(CONFIG["event_definitions"]), "pass": CONFIG["event_definitions"]["continuous_knock_in"]["separate_from"] == "discrete_autocall"},
    {"criterion": "8-12 deterministic cash-flow paths", "observed": f"{len(deterministic_results)} paths; outcome_pass={int(deterministic_results['outcome_pass'].sum())}", "pass": 8 <= len(deterministic_results) <= 12 and deterministic_results["outcome_pass"].all()},
    {"criterion": "component values sum to total value", "observed": f"max_abs_error={deterministic_results['identity_error'].abs().max():.3e}", "pass": deterministic_results["identity_error"].abs().max() <= 1e-10},
    {"criterion": "fair coupon root recovers known solution", "observed": f"known={known_coupon:.8f}; recovered={recovered_coupon:.8f}; error={root_error:+.3e}", "pass": abs(root_error) <= 1e-10},
    {"criterion": "valuation-scope exclusions recorded", "observed": "; ".join(CONFIG["valuation_scope_exclusions"]), "pass": len(CONFIG["valuation_scope_exclusions"]) == 4},
])
day1_pass = bool(gate_summary["pass"].all())
gate_status = "PASS" if day1_pass else "FAIL"
display(gate_summary)
print(f"Day 1 gate: {gate_status}")

component_identity_checks = deterministic_results[
    ["path_id", "contract", "total_value", "additive_sum", "identity_error"]
].copy()
run_manifest = pd.DataFrame([{
    "run_date": pd.Timestamp.now().isoformat(),
    "gate_status": gate_status,
    "project_root": str(PROJECT_DIR),
    "config": str(CONFIG_FILE.relative_to(PROJECT_DIR)),
    "config_schema_version": CONFIG["schema_version"],
    "workbook": str(SOURCE_FILE.relative_to(PROJECT_DIR)),
    "workbook_sha256": SOURCE_HASH,
    "deterministic_paths": len(deterministic_results),
    "component_identity_max_abs_error": deterministic_results["identity_error"].abs().max(),
    "known_coupon": known_coupon,
    "recovered_coupon": recovered_coupon,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
}])

layer_summary.to_csv(OUTPUT_DIR / "contract_layer_summary.csv", index=False)
deterministic_results.to_csv(OUTPUT_DIR / "deterministic_path_results.csv", index=False)
component_identity_checks.to_csv(OUTPUT_DIR / "component_identity_checks.csv", index=False)
fair_coupon_root_test.to_csv(OUTPUT_DIR / "fair_coupon_root_test.csv", index=False)
gate_summary.to_csv(OUTPUT_DIR / "gate_summary.csv", index=False)
run_manifest.to_csv(OUTPUT_DIR / "run_manifest.csv", index=False)
print(f"Saved Day 1 evidence to {OUTPUT_DIR}")

## Takeaways

- The issuer HSBC market case and the two research contracts now have
  separate machine-readable identities and claim boundaries.
- Continuous KI, discrete autocall, bump policy, fair-coupon accrual and
  valuation exclusions are explicit rather than implicit notebook choices.
- The component schema has a tested additive identity, while no-KI maturity
  redemption and event probabilities remain diagnostics rather than double-counted
  cash-flow components.
- The deterministic paths are unit tests for contract logic; they are not a
  calibration or a market-value result.